# LangGraph: Human-in-the-Loop with `interrupt`

**Self-learning notebook** — pause an agent mid-run, ask a human for help, then **resume** exactly where it left off.

## What you will learn

| Concept | Purpose |
|---------|---------|
| **`interrupt()`** | Pause graph execution inside a tool and surface a payload to the caller |
| **`Command(resume=...)`** | Feed the human's answer back and continue the same thread |
| **Human-as-tool** | Model can escalate to a person when it lacks expertise |
| **Checkpointer** | Required so paused state survives between `stream()` calls |

## Flow at a glance

```
User message → LLM → (calls human_assistance tool) → INTERRUPT
                                                          ↓
Human reviews query ←───────────────────────────────────┘
                                                          ↓
Command(resume={"data": "..."}) → tool returns → LLM → final answer
```

## Prerequisites

- Python 3.10+, LangGraph, LangChain
- Colab secrets (or `.env` locally):
  - `GROQ_API_KEY`
  - `TAVILY_API_KEY`

> **Companion notebook:** `LangGraph_Agents_Tools_Memory_Demo.ipynb` covers basic agents, tools, and memory.

In [ ]:
# Groq chat model integration
!pip install langchain-groq


## 1. API Keys & LLM Setup

We use Groq's Llama 3.3 70B via `init_chat_model` — a provider-agnostic LangChain helper.

In [ ]:
from google.colab import userdata

# Colab: add GROQ_API_KEY under Settings → Secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')


In [ ]:
from langchain.chat_models import init_chat_model

# Primary LLM for the agent
llm = init_chat_model(
    "groq:llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)
llm


## 2. Install LangGraph & Tavily

- **LangGraph** — graph orchestration + `interrupt` / `Command`
- **langchain_tavily** — optional web search tool alongside human assistance

In [ ]:
# Setup: install dependencies for this section (run once per session)
!pip install langchain_tavily


In [ ]:
# Setup: install dependencies for this section (run once per session)
!pip install langgraph langchain_core


In [ ]:
from typing import Annotated, TypedDict

from langchain_tavily import TavilySearch
from langchain_core.tools import tool as tool_decorator

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# interrupt pauses the graph; Command(resume=...) continues it
from langgraph.types import Command, interrupt


In [ ]:
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')


## 3. Build the Agent Graph

### The human-assistance tool

`interrupt({"query": query})` **stops** execution and returns control to your Python code. The graph stays paused until you call `graph.stream(Command(resume={"data": ...}))` with the same `thread_id`.

### Why disable parallel tool calls?

When resuming after an interrupt, parallel tool calls could re-fire tools that already ran. The chatbot node returns a single message to keep execution deterministic.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)


@tool_decorator
def human_assistance(query: str) -> str:
    """Request assistance from a human when the model needs expert input."""
    # Execution pauses here — the payload is shown to the human reviewer
    human_response = interrupt({"query": query})
    return human_response["data"]


tavily_tool = TavilySearch(max_results=2, tavily_api_key=TAVILY_API_KEY)
tools = [tavily_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)


def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    # Single message avoids duplicate tool calls when resuming after interrupt
    return {"messages": [message]}


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")  # ReAct loop


## 4. Compile with a Checkpointer

A **checkpointer is mandatory** for human-in-the-loop: it stores the paused graph state so the second `stream()` call can resume the same run.

In [ ]:
memory = MemorySaver()

graph = graph_builder.compile(checkpointer=memory)


In [ ]:
from IPython.display import display, Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Optional: needs pygraphviz/graphviz in some environments
    pass


## 5. Step 1 — Run Until Interrupt

Ask the agent to request human help. The graph will:

1. Route to the `human_assistance` tool
2. Hit `interrupt()` and **stop**
3. Print the tool's query for you to review

Watch the last message — it should be a **tool call** asking for human assistance, not a final answer.

In [ ]:
user_input = (
    "I need some expert guidance and assistance for building an AI agent. "
    "Could you request assistance for me?"
)
config = {"configurable": {"thread_id": 1}}

events = graph.stream(
    {"messages": user_input},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()


## 6. Step 2 — Resume with Human Input

Provide the expert answer via `Command(resume={"data": ...})`. The paused tool completes, returns the string to the LLM, and the graph continues to a final response.

**Important:** Use the **same `config` / `thread_id`** as the interrupted run.

In [ ]:
human_response = (
    "We the experts are here to help! We'd recommend you check out LangGraph to build your agent. "
    "It's much more reliable and extensible than simple autonomous agents."
)

# Resume the interrupted tool with the human's answer
human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()


---

## Key Takeaways

1. **`interrupt()`** turns any tool into a pause point — ideal for approvals, expert review, or missing data.
2. **`Command(resume=...)`** is how you inject the human's response and unblock the graph.
3. **Same `thread_id`** links the pause and resume calls into one logical run.
4. **Checkpointer required** — without it, paused state cannot be restored.

## Try on your own

- Change the user prompt so the model uses **Tavily** instead of human help.
- Add a second `interrupt` for **approval before** running Tavily search.
- Build a simple UI that shows `interrupt` payloads and collects human input.

## Next Steps

- [LangGraph human-in-the-loop docs](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)
- Persistent checkpointers (Postgres) for production pause/resume across server restarts